## 0. Convert CSV to long format with metrics grouped by date

In [ ]:
import pandas as pd
from pathlib import Path
from layoff.config import MERGED_DATA

def income_statement_to_dict_df(file, statement_name):
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    rows = []

    # First column contains metric names
    metric_col = df.columns[0]

    # Remaining columns are dates
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

## 1. Extract all the companies

In [15]:
from pathlib import Path
from layoff.config import DATA_PROCESSED, DATA_BALANCE_SHEET, DATA_FINANCIALS, DATA_CASHFLOWS

DATA_TICKER = DATA_PROCESSED/ "layoffs_with_tickers.csv"
tickers = set()

for file in DATA_BALANCE_SHEET.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)
 
print(f"Found {len(tickers)} companies")

Found 2708 companies


## 2. Load and merge balance sheet, cash flow, and income statements for each company on company and date

In [ ]:
master_rows = []

for ticker in tickers:

    dfs = []

    FILE_CONFIGS = [
        ("balancesheet", DATA_BALANCE_SHEET, "_balancesheet.csv"),
        ("cashflow", DATA_CASHFLOWS, "_cashflow.csv"),
        ("financials", DATA_FINANCIALS, "_financials.csv"),
    ]

    for statement_name, directory, suffix in FILE_CONFIGS:

        file = directory / f"{ticker}{suffix}"

        if not file.exists():
            continue

        try:
            data_df  = income_statement_to_dict_df(
                file,
                statement_name
            )

            if not data_df.empty:
                dfs.append(data_df)

        except Exception as e:
            print(
                f"Failed {statement_name} "
                f"for {ticker}: {e}"
            )

    # Skip companies with no statements
    if len(dfs) == 0:
        continue

    # Merge all available statements
    company_df = dfs[0]

    for df in dfs[1:]:
        company_df = company_df.merge(
            df,
            on=["company", "date"],
            how="outer"
        )

    master_rows.append(company_df)
    print(f"Processed {ticker}")

Processed ASRT
Processed CAG
Processed LOGI
Processed MTA.V
Processed CACI
Processed CHRS
Processed ASIX
Processed NOVT
Processed BMI
Processed BLKB
Processed WAVE
Processed 9988.HK
Processed SU.PA
Processed 0008.KL
Processed CRWD
Processed 1CD.F
Processed TPR
Processed THC
Processed 852.SG
Processed SCCO
Processed FTNT
Processed TRAX
Processed ROCK.V
Processed 600223.SS
Processed CHGG
Processed CNA.L
Processed 301231.SZ
Processed UIO.F
Processed KELYA
Processed CMSQY
Processed HP
Processed BC
Processed CASY
Processed AA
Processed LVS
Processed BKKT
Processed HSTM
Processed PGIL.NS
Processed PEP
Processed PPC
Processed PACB
Processed 3529.T
Processed LONN.SW
Processed PUMP
Processed PGR
Processed VALMT.HE
Processed KVUE
Processed C06.SI
Processed HLT
Processed RGTI
Processed 0QZ3.IL
Processed 92J.F
Processed BRK.AX
Processed KO
Processed PTON
Processed APPF
Processed KCPSUGIND.BO
Processed FORM
Processed WWW
Processed V
Processed TEL
Processed WIX
Processed GWRE
Processed 075180.KS
Pro

## 3. Flatten nested financial metrics into a single feature table with prefixed columns per company-date

In [ ]:
import pandas as pd

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"])

dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

bs_features = pd.json_normalize(dataset["balancesheet"]).add_prefix("bs_")
cf_features = pd.json_normalize(dataset["cashflow"]).add_prefix("cf_")

print(master_rows)

fin_features = pd.json_normalize(dataset["financials"]).add_prefix("fin_")

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"]],
        bs_features,
        cf_features,
        fin_features,
    ],
    axis=1,
)

features_df.to_csv(MERGED_DATA["MERGED_OUTPUT_CSV_PATH"], index=False)

# Verify
print(features_df.shape)
print(features_df.head())

[  company        date                                       balancesheet  \
0    ASRT  2024-09-30  {'Net Debt': 551000.0, 'Other Non Current Asse...   
1    ASRT  2024-12-31  {'Non Current Prepaid Assets': 1000.0, 'Constr...   
2    ASRT  2025-03-31  {'Ordinary Shares Number': 6384872.0, 'Share I...   
3    ASRT  2025-06-30  {'Ordinary Shares Number': 6414482.0, 'Share I...   
4    ASRT  2025-09-30  {'Ordinary Shares Number': 6416518.0, 'Share I...   
5    ASRT  2025-12-31  {'Ordinary Shares Number': 6421899.0, 'Share I...   
6    ASRT  2026-03-31  {'Ordinary Shares Number': 6445161.0, 'Share I...   

                                            cashflow  \
0  {'Repayment Of Debt': 0.0, 'Income Tax Paid Su...   
1  {'Repayment Of Debt': 0.0, 'Income Tax Paid Su...   
2  {'Free Cash Flow': -12538000.0, 'Interest Paid...   
3  {'Free Cash Flow': 19091000.0, 'Interest Paid ...   
4  {'Free Cash Flow': -4767000.0, 'Interest Paid ...   
5  {'Free Cash Flow': -29968000.0, 'Interest Paid...  